In [ ]:
import json
import glob
import os
from datetime import datetime
from loguru import logger
import pandas as pd

log_level = "INFO"
logger.add("log/consolidacion.log", rotation="10 MB", level="INFO")


3

In [ ]:
def consolidar_archivos_json(patron="registros_*.json", 
                             directorio='raw_jsons/',
                             archivo_salida="dataset_desaparecidos.csv",
                            ):
    """
    Combina archivos JSON en uno solo y genera 2 versiones:
    con y sin duplicados por ID (IDvictimadirecta).
    Args:
        patron: regex de archivos a buscar (ej: "registros_*.json")
        directorio: folder donde se encuentran los datos. Puede ser None para buscar 
        en el directorio actual
        archivo_salida: Nombre del archivo consolidad
    """

    logger.info("Inciiando merge de jsons...")

    
    if directorio is None:
        archivos = sorted(glob.glob(patron))
    else:
        ruta_completa = os.path.join(directorio, patron)
        archivos = sorted(glob.glob(ruta_completa))
            
    if not archivos:
        logger.error(f"❌ No se encontraron archivos con el patrón: {patron}")
        return
    
    logger.info(f"Archivos encontrados: {len(archivos)}")
    
    todos_los_registros = []
    registros_por_archivo = {}
    total_leidos = 0
    registros_invalidos = 0
    archivos_problematicos = []
    
    logger.info("Leyendo archivos...")
    for archivo in archivos:
        try:
            nombre_archivo = os.path.basename(archivo)
            
            with open(archivo, 'r', encoding='utf-8') as f:
                datos = json.load(f)
                
                if isinstance(datos, list):
                    registros_validos = 0
                    registros_invalidos_archivo = 0

                    # quitar los primeros 10 registros, son intrusos que no tienen que ver con el dataset (landing page de la API)
                    datos = datos[10:]

                    for i, registro in enumerate(datos):
                        if isinstance(registro, dict):
                            registro_con_origen = registro.copy()
                            registro_con_origen['archivo_origen'] = nombre_archivo
                            
                            todos_los_registros.append(registro_con_origen)
                            registros_validos += 1
                        else:
                            registros_invalidos += 1
                            registros_invalidos_archivo += 1
                            #logger.warning(f"  {nombre_archivo}: Registro {i} no es un diccionario (tipo: {type(registro).__name__})")
                            
                    
                    registros_por_archivo[archivo] = registros_validos
                    total_leidos += registros_validos
                    
                    if registros_invalidos_archivo > 0:
                        archivos_problematicos.append({
                            'archivo': archivo,
                            'registros_invalidos': registros_invalidos_archivo,
                            'registros_validos': registros_validos
                        })
                        #logger.warning(f"{archivo}: {registros_validos:,} registros válidos, {registros_invalidos_archivo} inválidos")

                else:
                    #logger.warning(f"{archivo}: formato inesperado (no es lista)")
                    archivos_problematicos.append({
                        'archivo': archivo,
                        'error': 'No es una lista',
                        'tipo': type(datos).__name__
                    })
                    
        except json.JSONDecodeError as e:
            logger.error(f"{archivo}: Error de JSON - {e}")
            archivos_problematicos.append({
                'archivo': archivo,
                'error': f'JSONDecodeError: {e}'
            })
        except Exception as e:
            logger.error(f"{archivo}: Error - {e}")
            archivos_problematicos.append({
                'archivo': archivo,
                'error': str(e)
            })
    
    logger.info(f"Total de registros leídos: {total_leidos:,}")
    if registros_invalidos > 0:
        logger.warning(f"Registros inválidos encontrados: {registros_invalidos:,}")
    
    if not todos_los_registros:
        logger.error("No se encontraron registros válidos para consolidar")
        return
    
    logger.info("Convirtiendo a DataFrame...")
    try:
        df = pd.DataFrame(todos_los_registros)
        logger.success(f" DataFrame creado: {len(df):,} filas x {len(df.columns)} columnas")
    except Exception as e:
        logger.error(f" Error creando DataFrame: {e}")
        return

    # ordenar por fecha si existe el campo
    logger.info("Ordenando datos...")
    try:
        if 'fechahechos' in df.columns:
            df = df.sort_values(by='fechahechos')
            logger.success(f"Registros ordenados por fechahechos")
        else:
            logger.warning(f"Campo 'fechahechos' no encontrado")
    except Exception as e:
        logger.warning(f"No se pudo ordenar: {e}")
    
    # guardar archivo CON duplicados
    archivo_con_duplicados = archivo_salida.replace('.csv', '_con_duplicados.csv')
    logger.info(f"Guardando archivo CON duplicados: {archivo_con_duplicados}..")
    try:
        df.to_csv(archivo_con_duplicados, index=False, encoding='utf-8-sig')
        tamaño_con_dup = os.path.getsize(archivo_con_duplicados) / (1024 * 1024)
        logger.success(f"Archivo guardado ({tamaño_con_dup:.2f} MB)")
    except Exception as e:
        logger.error(f" Error guardando archivo: {e}")
        return
    

    logger.info("Eliminando duplicados...")
    original = len(df)
    
    if 'IDvictimadirecta' in df.columns:
        df_sin_dup = df.drop_duplicates(subset=['IDvictimadirecta'], keep='first')
        logger.info(f"Duplicados eliminados por 'IDvictimadirecta': {original - len(df_sin_dup):,}")
        logger.info(f"Se mantiene la primera ocurrencia (con su archivo_origen)")
    else:
        logger.warning(f"Campo 'IDvictimadirecta' no encontrado, eliminando duplicados totales")
        df_sin_dup = df.drop_duplicates(keep='first')
        logger.info(f"Duplicados eliminados: {original - len(df_sin_dup):,}")
    
    logger.info(f"Registros únicos: {len(df_sin_dup):,}")
    

    archivo_sin_duplicados = archivo_salida.replace('.csv', '_sin_duplicados.csv')
    logger.info(f"Guardando archivo SIN duplicados: {archivo_sin_duplicados}")
    try:
        df_sin_dup.to_csv(archivo_sin_duplicados, index=False, encoding='utf-8-sig')
        tamaño_sin_dup = os.path.getsize(archivo_sin_duplicados) / (1024 * 1024)
        logger.success(f"Archivo guardado ({tamaño_sin_dup:.2f} MB)")
    except Exception as e:
        logger.error(f"Error guardando archivo: {e}")
        return

    # # estadísticas de archivos de origen
    # logger.debug("Distribución por archivo de origen:")
    # logger.debug("ANTES de eliminar duplicados globales:")
    # distribucion_archivos_original = df['archivo_origen'].value_counts().sort_index()
    # for archivo, cantidad in distribucion_archivos_original.items():
    #     logger.debug(f"{archivo}: {cantidad:,} registros")
    
    # logger.debug("DESPUÉS de eliminar duplicados globales:")
    # distribucion_archivos = df_sin_dup['archivo_origen'].value_counts().sort_index()
    # for archivo, cantidad in distribucion_archivos.items():
    #     registros_originales = distribucion_archivos_original.get(archivo, 0)
    #     duplicados_globales = registros_originales - cantidad
    #     logger.debug(f"{archivo}: {cantidad:,} registros únicos ({duplicados_globales:,} eran duplicados de otros archivos)")


    # resumen final
    logger.success("CONSOLIDACIÓN COMPLETADA")
    logger.info(f"Archivos procesados: {len(archivos)}")
    if archivos_problematicos:
        logger.warning(f"Archivos con problemas: {len(archivos_problematicos)}")
    logger.info(f"Registros válidos: {total_leidos:,}")
    if registros_invalidos > 0:
        logger.warning(f"Registros inválidos: {registros_invalidos:,}")
    logger.info(f"Registros únicos: {len(df_sin_dup):,}")
    logger.info(f"Duplicados eliminados: {original - len(df_sin_dup):,}")

    
    # Mostrar archivos problemáticos si los hay
    if archivos_problematicos:
        logger.warning("\nARCHIVOS CON PROBLEMAS DETECTADOS:")
        for problema in archivos_problematicos:
            logger.warning(f"{problema}")
    
    return df, df_sin_dup


In [ ]:
df_dup, df_sin_dup = consolidar_archivos_json(
            patron="registros_*.json",#"registros_*.json",
            directorio='clean_jsons/',
            archivo_salida="datos/dataset_desaparecidos_v2.csv"
        )

2026-05-15 20:18:51.671 | INFO     | __main__:consolidar_archivos_json:15 - Inciiando merge de jsons...
2026-05-15 20:18:51.673 | INFO     | __main__:consolidar_archivos_json:28 - Archivos encontrados: 148
2026-05-15 20:18:51.675 | INFO     | __main__:consolidar_archivos_json:36 - Leyendo archivos...
2026-05-15 20:18:55.135 | INFO     | __main__:consolidar_archivos_json:96 - Total de registros leídos: 248,982
2026-05-15 20:18:55.137 | WARNING  | __main__:consolidar_archivos_json:98 - Registros inválidos encontrados: 1,907
2026-05-15 20:18:55.138 | INFO     | __main__:consolidar_archivos_json:104 - Convirtiendo a DataFrame...
2026-05-15 20:18:55.988 | SUCCESS  | __main__:consolidar_archivos_json:107 -  DataFrame creado: 248,982 filas x 23 columnas
2026-05-15 20:18:55.988 | INFO     | __main__:consolidar_archivos_json:113 - Ordenando datos...
2026-05-15 20:18:56.430 | SUCCESS  | __main__:consolidar_archivos_json:117 - Registros ordenados por fechahechos
2026-05-15 20:18:56.432 | INFO    

In [17]:
df_dup[df_dup['IDvictimadirecta'] == 'CONFIDENCIAL']['archivo_origen'].value_counts()

archivo_origen
registros_2024-05-01_2024-05-31.json    839
registros_2024-01-01_2024-01-31.json    825
registros_2024-04-01_2024-04-30.json    789
registros_2024-02-01_2024-02-29.json    753
registros_2025-01-01_2025-01-31.json    625
registros_2024-03-01_2024-03-31.json    600
registros_2026-04-01_2026-04-30.json    544
registros_2026-03-01_2026-03-31.json    538
registros_2026-02-01_2026-02-28.json    453
registros_2026-01-01_2026-01-31.json    375
registros_2026-05-01_2026-05-31.json    302
Name: count, dtype: int64